# Session 1: What a machine means when it says it learns

**259.075 Data-integrated Algorithmic Design Processes II** · Session 1 of 7

<img src="img/wien-zinshaus.jpg" width="620">

*A late nineteenth century apartment house in Vienna, Lange Gasse 25. Photo: Funke,
[Wikimedia Commons](https://commons.wikimedia.org/wiki/File:1080_Lange_Gasse_25.jpg),
CC BY-SA 4.0. Eight hundred and fifty of the 1073 buildings in our table are of this kind.*

Take a building in Vienna, say a corner house in the seventh district, and ask what it needs
for heating. The usual way to answer takes an afternoon: you enter the geometry, the wall
build-ups, the windows, you run a simulation, you get a number. Do that for a hundred design
variants and you have spent a month.

> A model is a fast answer to a question that used to be slow.

We spend this term building something that answers in a millisecond instead, well enough to
be useful while you are still drawing. Today we build the smallest such thing there is: one
number in, one number out, and a rule in between that the machine finds by itself.

A word on the data before we start. The buildings are real, from the open data of the City of
Vienna, layer `GEBAEUDETYPOGD`, one polygon per building with its construction period. The
heating demand is not. No open dataset in Vienna gives measured consumption per building, so
the target value in this table comes from a formula combining the wall U-value of the
construction period with the shape of the footprint. That matters more than it sounds, and we
return to it at the end.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

df = pd.read_csv("../data/gebaeude_wien.csv")
print(df.shape)
df.head()

## Guessing first

Run the next cell. One point per building: compactness on the horizontal axis, heating demand
on the vertical.

Compactness is the perimeter of the footprint divided by the square root of its area. A round
building sits at 3.5, a long slab far above. Hover over three or four points and see what kind
of houses they are.

> Compactness: perimeter over square root of area. Dimensionless, so a shed and a block can be
> compared.

One question you should be able to answer before reading on: why do the points not lie on a
line, although the demand comes from a formula?

In [ ]:
fig = px.scatter(df, x="kompaktheit", y="hwb", color="startjahr",
                 hover_data=["id", "bauperiode", "bautyp", "flaeche_m2", "u_wert"],
                 labels={"kompaktheit": "compactness (perimeter / sqrt area)",
                         "hwb": "heating demand, computed [kWh/m2a]",
                         "startjahr": "period from"},
                 color_continuous_scale="Viridis", height=520)
fig.update_traces(marker=dict(size=7, opacity=0.75))
fig.show()

The same houses on the map. Find the one you live in, or the institute, click it and keep the
id. You will meet it again at the end of the session.

In [ ]:
fig = px.scatter_map(df, lat="lat", lon="lon", color="hwb", size="flaeche_m2",
                     color_continuous_scale="Turbo", zoom=11, height=520,
                     hover_data=["id", "bauperiode", "kompaktheit", "u_wert"],
                     map_style="carto-positron")
fig.show()

## Your line, by hand

Run the cell and drag the slider. The line `hwb = w · compactness + b` moves with it, the
intercept stays fixed for now, and the title shows one number: the mean squared error, the
average of the squared vertical distances between line and points.

Find the smallest value you can and write it down. We need it in twenty minutes.

<img src="img/linear-regression.png" width="460">

*What you are doing by hand is two hundred years old. Legendre and Gauss published it around
1805 as the method of least squares. Figure: Sewaqu,
[Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Linear_regression.svg), public
domain.*

In [ ]:
def mse(w, b, x, y):
    return float(np.mean((w * x + b - y) ** 2))

x = df["kompaktheit"].to_numpy()
y = df["hwb"].to_numpy()
xs = np.linspace(x.min(), x.max(), 50)
b_fix = 110.0
ws = np.arange(0, 31, 1)

frames = [go.Frame(name=f"{w}",
                   data=[go.Scatter(x=xs, y=w * xs + b_fix, mode="lines",
                                    line=dict(width=4, color="crimson"))],
                   traces=[1],
                   layout=go.Layout(title=f"w = {w},  b = {b_fix:.0f},  error = {mse(w, b_fix, x, y):.0f}"))
          for w in ws]

fig = go.Figure(
    data=[go.Scatter(x=x, y=y, mode="markers", name="buildings",
                     marker=dict(size=6, opacity=0.5, color="#4c78a8")),
          go.Scatter(x=xs, y=ws[0] * xs + b_fix, mode="lines", name="your model",
                     line=dict(width=4, color="crimson"))],
    frames=frames)
fig.update_layout(
    height=560, xaxis_title="compactness", yaxis_title="heating demand, computed",
    title=f"w = {ws[0]},  b = {b_fix:.0f},  error = {mse(ws[0], b_fix, x, y):.0f}",
    sliders=[dict(active=0, currentvalue=dict(prefix="w = "),
                  steps=[dict(method="animate", label=f"{w}",
                              args=[[f"{w}"], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
                         for w in ws])])
fig.show()

## The neuron

The line you were moving is

    hwb = w · compactness + b

and in the language of the field this is a *neuron*: inputs, one weight per input, a bias
term, an output. Nothing more. Every network you will meet later, including the ones that draw
pictures, is built from this.

Squaring the distances punishes a few large mistakes more than many small ones. That is a
choice, not a law, and we meet a different one in session 6.

## Downhill in the dark

Move the slider two steps to the right and watch the error. If it grows you went the wrong
way, if it falls keep going. That rule is enough to find the minimum, and it needs no
knowledge of the shape of the landscape, only of whether it rises or falls where you stand.

> The slope of the error at your position has a name: the gradient. It tells you which way is
> downhill, nothing else.

<img src="img/gradient-descent.gif" width="420">

*The same idea on a two-dimensional landscape: the point keeps stepping downhill until it sits
at the bottom. Animation: Jacopo Bertolotti,
[Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Gradient_descent.gif), CC0.*

The next two cells draw that landscape, first over `w` alone, then as a surface over `w` and
`b`. Turn the surface until you see the valley from the side. It is narrow and it lies at an
angle, and that costs us dearly in a minute.

In [ ]:
w_grid = np.linspace(0, 30, 200)
losses = [mse(w, b_fix, x, y) for w in w_grid]

fig = go.Figure(go.Scatter(x=w_grid, y=losses, mode="lines", line=dict(width=3)))
fig.update_layout(height=420, xaxis_title="w", yaxis_title="mean squared error",
                  title=f"error over w, at fixed b = {b_fix:.0f}")
fig.show()

In [ ]:
W, B = np.meshgrid(np.linspace(0, 30, 60), np.linspace(60, 170, 60))
Z = np.array([[mse(w, b, x, y) for w in W[0]] for b in B[:, 0]])

fig = go.Figure(go.Surface(x=W[0], y=B[:, 0], z=Z, colorscale="Viridis", opacity=0.92))
fig.update_layout(height=620, title="error landscape over w and b",
                  scene=dict(xaxis_title="w", yaxis_title="b", zaxis_title="error"))
fig.show()

## Your part

The gradient for this model is given below. The loop is yours: two lines, one step against the
gradient for `w`, one for `b`.

The derivation is in the assignment sheet as a collapsible section. Open it if you want it, it
is not examined. If you are stuck for ten minutes, the solution is released, see
`kursregeln.md`; nobody loses the actual task over a bracket.

In [ ]:
def gradient(w, b, x, y):
    fehler = w * x + b - y
    return float(2 * np.mean(fehler * x)), float(2 * np.mean(fehler))


def fit(x, y, lernrate=0.015, schritte=3000, w0=0.0, b0=None):
    w, b = w0, float(np.mean(y)) if b0 is None else b0
    verlauf = [(w, b, mse(w, b, x, y))]
    for _ in range(schritte):
        gw, gb = gradient(w, b, x, y)

        # YOUR TURN: one step against the gradient, two lines, one for w and one for b.
        # The step length is the learning rate.
        raise NotImplementedError("update rule missing")

        verlauf.append((w, b, mse(w, b, x, y)))
    return w, b, np.array(verlauf)


w_end, b_end, verlauf = fit(x, y)
print(f"w = {w_end:.1f}, b = {b_end:.1f}, error = {verlauf[-1, 2]:.1f}")

In [ ]:
fig = go.Figure(go.Scatter(y=verlauf[:, 2], mode="lines", line=dict(width=3)))
fig.update_layout(height=380, xaxis_title="step", yaxis_title="error", yaxis_type="log",
                  title=f"loss curve, learning rate 0.015, final error {verlauf[-1, 2]:.1f}")
fig.show()

Left: the contours of the error landscape with the path the parameters take. Right: the line
settling into the cloud. Press play.

In [ ]:
from plotly.subplots import make_subplots

bilder = np.linspace(0, len(verlauf) - 1, 60).astype(int)
fig = make_subplots(rows=1, cols=2, subplot_titles=("path through the error landscape",
                                                    "model in the data"))
fig.add_trace(go.Contour(x=W[0], y=B[:, 0], z=Z, colorscale="Viridis", showscale=False),
              row=1, col=1)
fig.add_trace(go.Scatter(x=[verlauf[0, 0]], y=[verlauf[0, 1]], mode="lines+markers",
                         marker=dict(size=7, color="crimson"), name="parameters"), row=1, col=1)
fig.add_trace(go.Scatter(x=x, y=y, mode="markers", name="buildings",
                         marker=dict(size=5, opacity=0.4, color="#4c78a8")), row=1, col=2)
fig.add_trace(go.Scatter(x=xs, y=verlauf[0, 0] * xs + verlauf[0, 1], mode="lines",
                         line=dict(width=4, color="crimson"), name="model"), row=1, col=2)

fig.frames = [go.Frame(name=str(i),
                       data=[go.Scatter(x=verlauf[:i + 1, 0], y=verlauf[:i + 1, 1],
                                        mode="lines+markers",
                                        marker=dict(size=5, color="crimson")),
                             go.Scatter(x=xs, y=verlauf[i, 0] * xs + verlauf[i, 1],
                                        mode="lines", line=dict(width=4, color="crimson"))],
                       traces=[1, 3])
              for i in bilder]
fig.update_layout(height=560, showlegend=False,
                  updatemenus=[dict(type="buttons", x=0.02, y=1.15,
                                    buttons=[dict(label="Play", method="animate",
                                                  args=[None, dict(frame=dict(duration=60, redraw=True),
                                                                   fromcurrent=True)]),
                                             dict(label="Stop", method="animate",
                                                  args=[[None], dict(mode="immediate")])])])
fig.update_xaxes(title_text="w", row=1, col=1)
fig.update_yaxes(title_text="b", row=1, col=1)
fig.update_xaxes(title_text="compactness", row=1, col=2)
fig.update_yaxes(title_text="heating demand", row=1, col=2)
fig.show()

## Now break it

Change one number: raise the learning rate until the curve goes up instead of down. Both runs
go into your submission, together with one sentence on how you tell them apart.

> If the loss curve rises, the learning rate is too large. This is the most common failure in
> the whole field.

Three thousand steps for a straight line is embarrassing, and the reason is the narrow valley
you turned around earlier. If you finish early: compute the exact solution by least squares,
compare, and think about why the loop took so long. We deal with it in session 2.

In [ ]:
_, _, wild = fit(x, y, lernrate=0.04, schritte=60)

fig = go.Figure(go.Scatter(y=wild[:, 2], mode="lines+markers", line=dict(width=3, color="crimson")))
fig.update_layout(height=380, xaxis_title="step", yaxis_title="error", yaxis_type="log",
                  title="loss curve, learning rate 0.04")
fig.show()

## What the model actually learned

The error stops at about 50 and no amount of patience brings it lower. That is not the fault of the
loop. A building has more to it than the shape of its footprint, and the thing that matters
most here, the U-value of the wall, sits in the data but not in the model. Session 2 hands it
over and the error collapses.

Now the map again, coloured by residual: red where the model overestimates, blue where it
underestimates. Find the house you picked at the beginning. You will see a pattern by
construction period, and you should, because the target was computed from the construction
period in the first place.

> Our model recovers a calculation rule that someone wrote down. It has not learned how a house
> behaves. Everything in this course stands or falls on that difference.

This is deliberate: a rule has no measurement error, so everything in the first weeks can be
checked exactly. From session 3 we add noise, and from there telling signal from accident is
the whole problem.

In [ ]:
df["vorhersage"] = w_end * x + b_end
df["residuum"] = df["hwb"] - df["vorhersage"]

fig = px.scatter_map(df, lat="lat", lon="lon", color="residuum", zoom=11, height=560,
                     color_continuous_scale="RdBu", range_color=(-30, 30),
                     hover_data=["id", "bauperiode", "u_wert", "kompaktheit", "vorhersage", "hwb"],
                     map_style="carto-positron")
fig.show()

## Tests

This cell has to run green before you submit. It recomputes what your `fit` returns and
compares it with the best line there is for this data. These are the same tests the grader
uses, there with different buildings.

In [ ]:
w_t, b_t, verlauf_t = fit(x, y)

eigener_fehler = float(np.mean((w_t * x + b_t - y) ** 2))
bestes_w, bestes_b = np.polyfit(x, y, 1)
bester_fehler = float(np.mean((bestes_w * x + bestes_b - y) ** 2))

assert np.isfinite([w_t, b_t]).all(), "parameters ran away"
assert abs(eigener_fehler - verlauf_t[-1, 2]) < 1e-6, "reported error does not match the parameters"
assert verlauf_t[-1, 2] < verlauf_t[0, 2], "the error must fall during the run"
assert eigener_fehler < 1.05 * bester_fehler, (
    f"still too far off: {eigener_fehler:.1f} against {bester_fehler:.1f} at best")
print(f"all green, error {eigener_fehler:.2f}, best possible {bester_fehler:.2f}")